# Chatterbox TTS + Turbo — Colab runner

Runs the app on a free Colab GPU. Nothing installs on your own machine.

**Per session:** Runtime → *Change runtime type* → GPU, then Runtime → *Run all*.
First run takes ~3–5 min (pip) + ~1 min (Audacity, for the Script to Voice tab's
mastering step) + ~2–4 min (model download from Hugging Face).
The `https://xxxx.gradio.live` link appears in the last cell's output.

In [ ]:
# 1. Confirm a GPU is attached
!nvidia-smi -L || echo 'NO GPU — Runtime > Change runtime type > GPU, then rerun'

In [ ]:
# 2. Get the code from GitHub (clone first time, pull afterwards)
import os
REPO_URL = 'https://github.com/ducp507/chatterbox-colab.git'  # <-- change if you rename the repo
REPO_DIR = REPO_URL.rstrip('/').split('/')[-1][:-4]
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL}
else:
    !cd {REPO_DIR} && git pull --ff-only
%cd {REPO_DIR}

In [ ]:
# 3. Install deps (uses Colab's preinstalled torch/torchaudio/numpy)
!pip install -q -r requirements-colab.txt
# If you hit an import error below, run instead:  !pip install -q -r requirements.txt

In [ ]:
# 4. Install real Audacity + a virtual display (headless), for the
#    'Script to Voice' tab's mastering step (Compressor -> Normalize -> EQ).
#    ~1 min, once per Colab session. Skip this cell if you'll only use the
#    other tabs / always leave 'Chuẩn hoá bằng Audacity' unchecked.
!apt-get -qq update && apt-get install -y -qq audacity xvfb

In [ ]:
# 5. (optional) cache model weights on Google Drive so future sessions skip the re-download
#    Uncomment the 3 lines, approve the Drive popup once.
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ['HF_HOME'] = '/content/drive/MyDrive/hf-cache'
# os.makedirs(os.environ['HF_HOME'], exist_ok=True)

In [ ]:
# 6. Launch — wait for the  Running on public URL: https://....gradio.live  line
!GRADIO_SHARE=1 TOKENIZERS_PARALLELISM=false python app.py

### Keep a cloned voice between sessions
Voices you clone land in `modules/voice_samples/` which is wiped when the VM recycles.
To keep one, run in a new cell (needs a GitHub token with repo write):
```python
!git config user.email you@example.com && git config user.name you
!git add modules/voice_samples/ && git commit -m 'add voice' && git push https://TOKEN@github.com/ducp507/chatterbox-colab.git
```
Or just download the `.wav` from the file browser on the left.

### If 'Script to Voice' mastering fails
The Status box will show Audacity's own error text (e.g. a rejected macro
parameter, or the scripting pipe never appearing because mod-script-pipe
needs enabling by hand once). Screenshot/copy that text — it says exactly
which step failed and why, which is what's needed to patch
`modules/audacity_bridge.py`. Meanwhile you still get the un-mastered voice
back, or you can uncheck 'Chuẩn hoá bằng Audacity' to skip that step.